In [1]:
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
from IPython.display import Video
import jax.numpy as jnp
import jax.lax as lax
import jax
from functools import partial

from typing import Callable, List

In [2]:
key = jax.random.key(123)
key, key_ = jax.random.split(key)

In [3]:

# Basic metropolis algorithm
@partial(jax.jit, static_argnums=(2,3))
def metropolis_kernel(proposal_key, acceptance_key, target, proposal, q):
    # Sample q' ~ proposal(q'|q)
    q_proposed = proposal(proposal_key, q) # q'
    acceptance_ratio = target(q_proposed)/target(q)
    return jax.lax.select(jax.random.uniform(acceptance_key) < acceptance_ratio, q_proposed, q)

def metropolis(key: jax.Array, target: Callable[[float], float], initial: jax.Array, proposal: Callable[[float], float], iterations=100_000):
    samples: List[float] = [initial]
    key, *proposal_keys = jax.random.split(key, num=iterations+1)
    key, *acceptance_keys = jax.random.split(key, num=iterations+1)

    for i in range(iterations):
        q = samples[-1] # q
        q_prime = metropolis_kernel(proposal_keys[i], acceptance_keys[i], target, proposal, q)
        samples += [q_prime]
    return jnp.array(samples)

In [4]:
class NormalProposal:
    def __init__(self, scale):
        self.scale = scale
    
    def __call__(self, key, q):
        z = jax.random.normal(key, q.shape)
        return z * self.scale + q

In [5]:
class DonutPDF():
    def __init__(self, radius=3, var=0.05):
        self.radius = radius
        self.var = var
    
    def __call__(self, x):
        r = jnp.linalg.norm(x)
        return jnp.exp((-(r - self.radius)**2)/self.var)
    
    @partial(jax.jit, static_argnums=(0,))
    def log_density(self, x):
        r = jnp.linalg.norm(x)
        return -((r - self.radius) ** 2) / self.var

    @partial(jax.jit, static_argnums=(0,))
    def grad_log_density(self, x):
        r = jnp.linalg.norm(x)
        return jnp.where(r == 0, jnp.zeros_like(x), 2 * x * (self.radius / r - 1) / self.var)

In [6]:
target = DonutPDF()
samples05 = metropolis(key, target, jnp.array([0,3], dtype=float), NormalProposal(0.05), iterations=10_000)
samples1 = metropolis(key, target, jnp.array([0,3], dtype=float), NormalProposal(1), iterations=10_000)

In [66]:
import jax.version


f, ax = plt.subplots(figsize=(16, 8), ncols=2)
ax[0].tick_params('both', which='major', labelsize=14)
ax[1].tick_params('both', which='major', labelsize=14)

points = jnp.linspace(-4, 4, 301)
x = jnp.tile(points, len(points))
y = jnp.repeat(points, len(points))

z = jnp.exp(jnp.apply_along_axis(target, 1, jnp.column_stack([x, y])))

ax[0].contour(
    points,
    points,
    z.reshape(len(points), len(points)),
    cmap="plasma",
    alpha=0.3,
    levels=5,
)

ax[1].contour(
    points,
    points,
    z.reshape(len(points), len(points)),
    cmap="plasma",
    alpha=0.3,
    levels=5,
)

(traj05,) = ax[0].plot([], [])
(traj1,) = ax[1].plot([], [])

title05 = ax[0].set_title("Acceptance rate: ", fontsize=18)
title1 = ax[1].set_title("Acceptance rate: ", fontsize=18)

writer = FFMpegWriter(fps=25)

@jax.jit
def compare_mean(array1, array2):
    return 1 - jnp.all(array1 == array2, axis=1).mean()

with writer.saving(f, "donut_mcmc.mp4", 50):
    # initial empty frames
    for _ in range(25):
        writer.grab_frame()

    for i in range(2, 2500, 2):
        traj05.set_data(*(samples05[: i + 1].T))
        traj1.set_data(*(samples1[: i + 1].T))

        accept05 = 1 - (samples05[1 : i + 1] == samples05[:i]).all(axis=1).mean()
        accept1 = 1 - (samples1[1 : i + 1] == samples1[:i]).all(axis=1).mean()
        title05.set_text(f"Acceptance rate: {accept05 * 100:.1f}%")
        title1.set_text(f"Acceptance rate: {accept1 * 100:.1f}%")

        writer.grab_frame()

    # closing empty frames
    for _ in range(25):
        writer.grab_frame()

plt.savefig("metropolis-snapshot.png")
plt.close()

In [8]:
Video("donut_mcmc.mp4")

In [55]:
def leapfrog_step(q,p, acc, i, target,step_size):
    p = p + target.grad_log_density(q) * step_size / 2
    q = q + p * step_size
    p = p + target.grad_log_density(q) * step_size / 2
    return q, p, acc.at[i].set(q)

@partial(jax.jit, static_argnums=(2,3,4))
def leapfrog(q0, p0, target, L, step_size):
    return jax.lax.fori_loop(0,L, lambda i, x: leapfrog_step(x[0],x[1], x[2], i, target,step_size), (q0,p0, jnp.zeros((L, *q0.shape), dtype=float)))

In [56]:
leapfrog(jnp.array(0.2), jnp.array(3.2), target, 10, 1)

(Array(-9.227178e+15, dtype=float32),
 Array(1.750734e+17, dtype=float32),
 Array([ 5.9400002e+01, -2.1374001e+03,  8.1041805e+04, -3.0773312e+06,
         1.1685742e+08, -4.4375045e+09,  1.6850831e+11, -6.3988784e+12,
         2.4298887e+14, -9.2271778e+15], dtype=float32))

In [57]:
leapfrog(jnp.array(0.2), jnp.array(3.2), target, 10, 0.1)

(Array(0.312873, dtype=float32),
 Array(5.704072, dtype=float32),
 Array([1.08      , 2.728     , 4.4848    , 5.64768   , 5.7514877 ,
        4.7547007 , 3.0560334 , 1.334953  , 0.27989122, 0.312873  ],      dtype=float32))

In [62]:
@partial(jax.jit, static_argnums=(3,4,5))
def hmc_kernel(momentum_key, acceptance_key, q0, target, L, step_size):
    p0 = jax.random.normal(momentum_key, q0.size) # Sample momentum

    q_star, p_star, _ = leapfrog(q0, p0, target, L, step_size) # Integrate forward in phase space

    h0 = -target.log_density(q0) + (p0 * p0).sum() / 2 # Compute Hamiltonian energy from q
    h = -target.log_density(q_star) + (p_star * p_star).sum() / 2 # Compute Hamiltonian energy from q* (originally this uses a strange definition of hamiltonian energy)
    log_accept_ratio = h0 - h # in log space, so this is just acceptance ration
    return jnp.where(jax.random.uniform(acceptance_key) < jnp.exp(log_accept_ratio), q_star, q0) 

def hmc(key, target, initial, iterations=10_000, L=50, step_size=0.1):
    samples = [initial()]
    key, *momentum_keys = jax.random.split(key, num=iterations+1)
    key, *acceptance_keys = jax.random.split(key, num=iterations+1)
    for i in range(iterations):
        q0 = samples[-1]
        q_prime = hmc_kernel(momentum_keys[i], acceptance_keys[i], q0, target, L, step_size)
        samples.append(q_prime)

    return jnp.array(samples)

In [63]:
key, key_ = jax.random.split(key)
hmc_samples = hmc(key_, DonutPDF(), lambda: jnp.array([3.0, 0.0]))

In [67]:
f, ax = plt.subplots(figsize=(7, 7))

points = jnp.linspace(-4, 4, 101)
x = jnp.tile(points, len(points))
y = jnp.repeat(points, len(points))

donut = DonutPDF()
z = jnp.exp(jnp.apply_along_axis(donut.log_density, 1, jnp.column_stack([x, y])))

ax.contour(
    points,
    points,
    z.reshape(len(points), len(points)),
    cmap="plasma",
    alpha=0.5,
)

(samps,) = ax.plot([], [], "o", alpha=0.7)
(traj,) = ax.plot([], [], "k")

writer = FFMpegWriter(fps=15)

with writer.saving(f, "donut_hmc.mp4", 50):
    # initial empty frame
    writer.grab_frame()

    samples = [jnp.array([3.0, 0.0])]
    samps.set_data(*(jnp.array(samples).T))

    writer.grab_frame()

    for i in range(100):
        q0 = samples[-1]
        key, key_ = jax.random.split(key)
        p0 = 2*jax.random.normal(key_, q0.shape)

        q_star, p_star, trajectory = leapfrog(q0, p0, donut, 20, 0.1)

        h0 = -donut.log_density(q0) + (p0 * p0).sum() / 2
        h = -donut.log_density(q_star) + (p_star * p_star).sum() / 2
        log_accept_ratio = h0 - h

        # animate the trajectory
        for j in range(len(trajectory)):
            traj.set_data(*jnp.array(trajectory[: j + 1]).T)
            writer.grab_frame()

        # flash green or red to signal acceptance / rejection
        key, key_ = jax.random.split(key)
        if jax.random.uniform(key_) < jnp.exp(log_accept_ratio):
            samples.append(q_star)
            traj.set_color("tab:green")
        else:
            samples.append(q0)
            traj.set_color("tab:red")

        for _ in range(2):
            writer.grab_frame()

        traj.set_color("black")
        traj.set_data([], [])
        samps.set_data(*(jnp.array(samples).T))
        writer.grab_frame()

plt.savefig("hmc-snapshot.png")
plt.close()

In [65]:
Video("donut_hmc.mp4")